# Feature Engineering for Better Predictions\n\n**Chapter 6: Regression Techniques for Soccer Analytics - EXTRA**\n\n## Learning Objectives\n\n- Create interaction features to capture combined effects\n- Generate polynomial features for non-linear relationships\n- Encode categorical variables effectively\n- Build lagged features from temporal data\n- Create domain-specific soccer features\n- Understand feature selection techniques\n- Avoid common feature engineering pitfalls

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom sklearn.linear_model import LinearRegression\nfrom sklearn.preprocessing import PolynomialFeatures, StandardScaler\nfrom sklearn.model_selection import train_test_split, cross_val_score\nfrom sklearn.metrics import r2_score, mean_squared_error\nfrom sklearn.feature_selection import SelectKBest, f_regression, RFE\n\nsns.set_style('whitegrid')\nplt.rcParams['figure.figsize'] = (12, 8)

## Why Feature Engineering Matters\n\n**Quote:** \"Coming up with features is difficult, time-consuming, requires expert knowledge. 'Applied machine learning' is basically feature engineering.\" - Andrew Ng\n\n**In soccer analytics:**\n- Raw stats (shots, passes) are just the starting point\n- **Interaction effects** matter (e.g., shots × accuracy)\n- **Context** is crucial (home vs away, opponent strength)\n- **Temporal patterns** reveal form and momentum\n- **Domain knowledge** creates the best features

## Load Base Data

In [ ]:
# Generate realistic match data\nnp.random.seed(42)\nn_matches = 300\n\nteams = ['Arsenal', 'Chelsea', 'Liverpool', 'Man City', 'Man United', 'Tottenham', \n         'Leicester', 'West Ham', 'Everton', 'Wolves']\n\ndf = pd.DataFrame({\n    'match_id': range(n_matches),\n    'team': np.random.choice(teams, n_matches),\n    'opponent': np.random.choice(teams, n_matches),\n    'home': np.random.choice([0, 1], n_matches),\n    'shots': np.random.randint(5, 20, n_matches),\n    'shots_on_target': np.random.randint(2, 12, n_matches),\n    'possession': np.random.uniform(35, 70, n_matches),\n    'pass_accuracy': np.random.uniform(70, 92, n_matches),\n    'xg': np.random.uniform(0.5, 3.5, n_matches),\n    'opponent_xg': np.random.uniform(0.5, 3.5, n_matches),\n    'match_week': np.random.randint(1, 39, n_matches)\n})\n\n# Generate realistic goals\ndf['goals'] = (\n    df['xg'] * 0.8 + \n    df['shots_on_target'] * 0.05 + \n    df['home'] * 0.3 +\n    np.random.normal(0, 0.5, n_matches)\n).clip(0, 6).round().astype(int)\n\nprint(\"Base Dataset:\")\nprint(df.head())\nprint(f\"\\nShape: {df.shape}\")\nprint(f\"Features: {df.columns.tolist()}\")

## 1. Interaction Features\n\n**Concept:** Capture combined effects of two or more features.\n\n**Soccer examples:**\n- `shots × accuracy` = Quality of shooting\n- `possession × pass_accuracy` = Effective possession\n- `home × opponent_strength` = Home advantage against strong teams

In [ ]:
# Create interaction features\ndf['shot_quality'] = df['shots_on_target'] / (df['shots'] + 1)  # Avoid division by zero\ndf['effective_possession'] = df['possession'] * df['pass_accuracy'] / 100\ndf['xg_difference'] = df['xg'] - df['opponent_xg']\ndf['shot_efficiency'] = df['xg'] / (df['shots'] + 1)\ndf['attacking_threat'] = df['shots_on_target'] * df['xg']\n\nprint(\"Interaction Features Created:\")\nprint(df[['shot_quality', 'effective_possession', 'xg_difference', 'shot_efficiency', 'attacking_threat']].head())\n\n# Visualize impact\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\naxes[0].scatter(df['shots_on_target'], df['goals'], alpha=0.3, label='Shots on Target')\naxes[0].set_xlabel('Shots on Target')\naxes[0].set_ylabel('Goals')\naxes[0].set_title('Raw Feature')\naxes[0].legend()\n\naxes[1].scatter(df['attacking_threat'], df['goals'], alpha=0.3, color='green', label='Attacking Threat')\naxes[1].set_xlabel('Attacking Threat (SoT × xG)')\naxes[1].set_ylabel('Goals')\naxes[1].set_title('Interaction Feature')\naxes[1].legend()\n\nplt.tight_layout()\nplt.show()\n\nprint(\"\\nInteraction features often show stronger correlations with the target.\")

## 2. Polynomial Features\n\n**Concept:** Capture non-linear relationships by adding squared, cubed terms.\n\n**Example:** `xG²` might capture that very high xG matches are even more likely to produce goals.

In [ ]:
# Create polynomial features\npoly = PolynomialFeatures(degree=2, include_bias=False)\nfeatures_for_poly = ['xg', 'shots_on_target']\nX_poly = poly.fit_transform(df[features_for_poly])\n\n# Get feature names\npoly_feature_names = poly.get_feature_names_out(features_for_poly)\ndf_poly = pd.DataFrame(X_poly, columns=poly_feature_names)\n\nprint(\"Polynomial Features:\")\nprint(df_poly.head())\nprint(f\"\\nOriginal features: {len(features_for_poly)}\")\nprint(f\"Polynomial features: {len(poly_feature_names)}\")\nprint(f\"\\nNew features include: {list(poly_feature_names)}\")\n\n# Compare models\nX_original = df[features_for_poly]\ny = df['goals']\n\nX_train_orig, X_test_orig, y_train, y_test = train_test_split(X_original, y, test_size=0.25, random_state=42)\nX_train_poly, X_test_poly, _, _ = train_test_split(df_poly, y, test_size=0.25, random_state=42)\n\n# Linear model with original features\nmodel_orig = LinearRegression()\nmodel_orig.fit(X_train_orig, y_train)\nr2_orig = model_orig.score(X_test_orig, y_test)\n\n# Linear model with polynomial features\nmodel_poly = LinearRegression()\nmodel_poly.fit(X_train_poly, y_train)\nr2_poly = model_poly.score(X_test_poly, y_test)\n\nprint(f\"\\nModel Performance:\")\nprint(f\"  Original features R²: {r2_orig:.3f}\")\nprint(f\"  Polynomial features R²: {r2_poly:.3f}\")\nprint(f\"  Improvement: {(r2_poly - r2_orig)*100:.1f}%\")

## 3. Categorical Feature Encoding\n\n**Challenge:** Machine learning models need numbers, not text.\n\n**Methods:**\n- **Label Encoding:** Assign numbers (0, 1, 2...) - implies order\n- **One-Hot Encoding:** Create binary columns for each category\n- **Target Encoding:** Replace with mean target value\n- **Frequency Encoding:** Replace with category frequency

In [ ]:
# 1. Target Encoding (team strength based on average goals)\nteam_strength = df.groupby('team')['goals'].mean().to_dict()\nopponent_strength = df.groupby('opponent')['goals'].mean().to_dict()\n\ndf['team_strength'] = df['team'].map(team_strength)\ndf['opponent_strength'] = df['opponent'].map(opponent_strength)\n\nprint(\"Target Encoding - Team Strength:\")\nfor team, strength in sorted(team_strength.items(), key=lambda x: x[1], reverse=True)[:5]:\n    print(f\"  {team}: {strength:.2f} goals/match\")\n\n# 2. One-Hot Encoding for home/away\ndf['is_home'] = df['home']\ndf['is_away'] = 1 - df['home']\n\nprint(f\"\\nOne-Hot Encoding created: is_home, is_away\")\n\n# 3. Frequency Encoding\nteam_freq = df['team'].value_counts(normalize=True).to_dict()\ndf['team_frequency'] = df['team'].map(team_freq)\n\nprint(f\"\\nFrequency Encoding: Teams by match frequency\")\nprint(df.groupby('team')['team_frequency'].first().sort_values(ascending=False).head())

## 4. Lagged Features (Temporal)\n\n**Concept:** Use past performance to predict future outcomes.\n\n**Soccer examples:**\n- Last 3 matches average goals\n- Form (rolling average)\n- Momentum (trend)\n- Head-to-head history

In [ ]:
# Sort by team and match week\ndf = df.sort_values(['team', 'match_week']).reset_index(drop=True)\n\n# Create lagged features\ndf['goals_last_match'] = df.groupby('team')['goals'].shift(1)\ndf['goals_last_3_avg'] = df.groupby('team')['goals'].transform(\n    lambda x: x.rolling(window=3, min_periods=1).mean().shift(1)\n)\ndf['goals_last_5_avg'] = df.groupby('team')['goals'].transform(\n    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)\n)\n\n# Form indicator (recent performance)\ndf['recent_form'] = df['goals_last_3_avg'] - df['goals_last_5_avg']\n\nprint(\"Lagged Features:\")\nprint(df[['team', 'match_week', 'goals', 'goals_last_match', 'goals_last_3_avg', 'recent_form']].head(10))\n\n# Visualize form\nteam_example = 'Arsenal'\nteam_data = df[df['team'] == team_example].head(15)\n\nplt.figure(figsize=(12, 6))\nplt.plot(team_data['match_week'], team_data['goals'], 'o-', label='Actual Goals', linewidth=2)\nplt.plot(team_data['match_week'], team_data['goals_last_3_avg'], 's--', label='Last 3 Avg', linewidth=2)\nplt.xlabel('Match Week')\nplt.ylabel('Goals')\nplt.title(f'{team_example} - Goals and Form')\nplt.legend()\nplt.grid(True, alpha=0.3)\nplt.tight_layout()\nplt.show()\n\nprint(f\"\\nLagged features capture team form and momentum.\")

## 5. Domain-Specific Features\n\n**Soccer-specific features** based on tactical knowledge:

In [ ]:
# 1. Defensive solidity\ndf['defensive_solidity'] = 1 / (df['opponent_xg'] + 0.1)  # Lower opponent xG = better defense\n\n# 2. Attacking efficiency\ndf['attacking_efficiency'] = df['xg'] / (df['shots'] + 1)\n\n# 3. Possession quality\ndf['possession_quality'] = (df['possession'] * df['pass_accuracy']) / 100\n\n# 4. Match importance (later in season = more important)\ndf['match_importance'] = df['match_week'] / 38  # Normalize to 0-1\n\n# 5. Strength differential\ndf['strength_diff'] = df['team_strength'] - df['opponent_strength']\n\n# 6. Expected goal difference\ndf['expected_goal_diff'] = df['xg'] - df['opponent_xg']\n\nprint(\"Domain-Specific Features:\")\nprint(df[['defensive_solidity', 'attacking_efficiency', 'possession_quality', \n          'strength_diff', 'expected_goal_diff']].head())\n\n# Correlation with goals\nfeature_cols = ['defensive_solidity', 'attacking_efficiency', 'possession_quality', \n                'strength_diff', 'expected_goal_diff']\ncorrelations = df[feature_cols + ['goals']].corr()['goals'].drop('goals').sort_values(ascending=False)\n\nprint(f\"\\nCorrelation with Goals:\")\nfor feat, corr in correlations.items():\n    print(f\"  {feat}: {corr:.3f}\")

## 6. Feature Selection\n\n**Problem:** Too many features can cause overfitting.\n\n**Methods:**\n1. **Filter methods:** Select based on statistical tests\n2. **Wrapper methods:** Select based on model performance\n3. **Embedded methods:** Feature selection during training

In [ ]:
# Prepare feature set (drop NaN from lagged features)\ndf_clean = df.dropna()\n\nfeature_candidates = [\n    'shots', 'shots_on_target', 'possession', 'pass_accuracy', 'xg', 'opponent_xg',\n    'home', 'shot_quality', 'effective_possession', 'xg_difference', 'shot_efficiency',\n    'attacking_threat', 'team_strength', 'opponent_strength', 'goals_last_3_avg',\n    'recent_form', 'defensive_solidity', 'attacking_efficiency', 'strength_diff'\n]\n\nX_full = df_clean[feature_candidates]\ny_full = df_clean['goals']\n\nprint(f\"Total candidate features: {len(feature_candidates)}\")\n\n# 1. Filter Method: SelectKBest\nselector_filter = SelectKBest(score_func=f_regression, k=10)\nX_filtered = selector_filter.fit_transform(X_full, y_full)\nselected_features_filter = [feature_candidates[i] for i in selector_filter.get_support(indices=True)]\n\nprint(f\"\\nFilter Method (SelectKBest) - Top 10 features:\")\nfor i, feat in enumerate(selected_features_filter, 1):\n    print(f\"  {i}. {feat}\")\n\n# 2. Wrapper Method: Recursive Feature Elimination\nmodel_rfe = LinearRegression()\nselector_rfe = RFE(model_rfe, n_features_to_select=10, step=1)\nselector_rfe.fit(X_full, y_full)\nselected_features_rfe = [feature_candidates[i] for i in range(len(feature_candidates)) if selector_rfe.support_[i]]\n\nprint(f\"\\nWrapper Method (RFE) - Top 10 features:\")\nfor i, feat in enumerate(selected_features_rfe, 1):\n    print(f\"  {i}. {feat}\")

## Compare: All Features vs. Selected Features

In [ ]:
# Split data\nX_train_full, X_test_full, y_train, y_test = train_test_split(X_full, y_full, test_size=0.25, random_state=42)\nX_train_sel, X_test_sel, _, _ = train_test_split(X_full[selected_features_filter], y_full, test_size=0.25, random_state=42)\n\n# Model with all features\nmodel_all = LinearRegression()\nmodel_all.fit(X_train_full, y_train)\nr2_all = model_all.score(X_test_full, y_test)\nrmse_all = np.sqrt(mean_squared_error(y_test, model_all.predict(X_test_full)))\n\n# Model with selected features\nmodel_sel = LinearRegression()\nmodel_sel.fit(X_train_sel, y_train)\nr2_sel = model_sel.score(X_test_sel, y_test)\nrmse_sel = np.sqrt(mean_squared_error(y_test, model_sel.predict(X_test_sel)))\n\nprint(\"Model Comparison:\")\nprint(f\"\\nAll Features ({len(feature_candidates)}):\")\nprint(f\"  R²: {r2_all:.3f}\")\nprint(f\"  RMSE: {rmse_all:.3f}\")\nprint(f\"\\nSelected Features ({len(selected_features_filter)}):\")\nprint(f\"  R²: {r2_sel:.3f}\")\nprint(f\"  RMSE: {rmse_sel:.3f}\")\n\nif r2_sel >= r2_all * 0.98:  # Within 2%\n    print(f\"\\n✅ Feature selection maintained performance with {len(feature_candidates) - len(selected_features_filter)} fewer features!\")\nelse:\n    print(f\"\\n⚠️ Feature selection reduced performance. Consider keeping more features.\")

## Common Pitfalls to Avoid\n\n### 1. Data Leakage\n**Problem:** Using information that wouldn't be available at prediction time.\n\n**Example:** Including \"goals in next match\" as a feature 😱\n\n**Solution:** Only use features available before the match.

In [ ]:
# WRONG: Using future information\n# df['goals_next_match'] = df.groupby('team')['goals'].shift(-1)  # DON'T DO THIS!\n\n# RIGHT: Using only past information\ndf['goals_previous_match'] = df.groupby('team')['goals'].shift(1)  # OK\n\nprint(\"✅ Always ensure lagged features use .shift(positive_number)\")\nprint(\"❌ Never use .shift(negative_number) - that's future data!\")

### 2. Multicollinearity\n**Problem:** Highly correlated features can make models unstable.\n\n**Solution:** Remove or combine correlated features.

In [ ]:
# Check correlation matrix\ncorr_matrix = X_full.corr()\n\n# Find highly correlated pairs\nhigh_corr_pairs = []\nfor i in range(len(corr_matrix.columns)):\n    for j in range(i+1, len(corr_matrix.columns)):\n        if abs(corr_matrix.iloc[i, j]) > 0.8:\n            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))\n\nif high_corr_pairs:\n    print(\"⚠️ Highly Correlated Feature Pairs (|r| > 0.8):\")\n    for feat1, feat2, corr in high_corr_pairs:\n        print(f\"  {feat1} <-> {feat2}: {corr:.3f}\")\n    print(\"\\nConsider removing one feature from each pair.\")\nelse:\n    print(\"✅ No severe multicollinearity detected.\")

### 3. Overfitting with Too Many Features\n**Problem:** Model memorizes training data instead of learning patterns.\n\n**Solution:** Use cross-validation and regularization.

In [ ]:
# Compare training vs test performance\nfrom sklearn.model_selection import cross_val_score\n\nmodel = LinearRegression()\n\n# Cross-validation scores\ncv_scores = cross_val_score(model, X_full, y_full, cv=5, scoring='r2')\n\nprint(\"Cross-Validation Performance:\")\nprint(f\"  Mean R²: {cv_scores.mean():.3f}\")\nprint(f\"  Std R²: {cv_scores.std():.3f}\")\nprint(f\"\\nIf std is high (>0.1), model may be overfitting.\")\n\nif cv_scores.std() > 0.1:\n    print(\"⚠️ High variance detected. Consider:\")\n    print(\"  - Reducing number of features\")\n    print(\"  - Using regularization (Ridge/Lasso)\")\n    print(\"  - Collecting more data\")\nelse:\n    print(\"✅ Model shows stable performance across folds.\")

## Summary\n\nIn this notebook, we:\n\n1. ✅ Created interaction features for combined effects\n2. ✅ Generated polynomial features for non-linearity\n3. ✅ Encoded categorical variables multiple ways\n4. ✅ Built lagged features from temporal data\n5. ✅ Designed domain-specific soccer features\n6. ✅ Applied feature selection techniques\n7. ✅ Learned to avoid common pitfalls\n\n## Key Takeaways\n\n- **Feature engineering > Model selection** in most cases\n- **Interaction features** capture combined effects\n- **Polynomial features** handle non-linear relationships\n- **Lagged features** capture temporal patterns and form\n- **Domain knowledge** creates the most powerful features\n- **Feature selection** prevents overfitting\n- **Avoid data leakage** at all costs\n- **Check for multicollinearity** in your features\n\n## Feature Engineering Checklist\n\n✅ Create interaction terms for related features\n✅ Add polynomial terms if relationships are non-linear\n✅ Encode categorical variables appropriately\n✅ Include lagged features for temporal data\n✅ Design domain-specific features using soccer knowledge\n✅ Check for and remove highly correlated features\n✅ Use feature selection to reduce dimensionality\n✅ Validate no data leakage exists\n✅ Test on holdout data to ensure generalization

## Practice Exercises\n\n1. **Create More Interactions**: Try 3-way interactions (e.g., home × xG × opponent_strength)\n2. **Time-Based Features**: Add day of week, month, season period\n3. **Rolling Statistics**: Create rolling min/max, not just averages\n4. **Opponent-Specific Features**: Head-to-head history, tactical matchups\n5. **Feature Importance**: Use Random Forest to rank feature importance\n6. **Automated Feature Engineering**: Research and try automated tools like Featuretools\n7. **Real Data Application**: Apply these techniques to actual match data